# Supply Chain Data Analyst — тестовое задание (Генерация)

Расчёт рекомендуемого количества к заказу (`RecommendedOrder`) по 200 SKU на дату 19.09.2025.

## Настройка

Импорты и настраиваемые параметры — общие для всего ноутбука. Все новые импорты добавляются в ячейку ниже, все новые параметры — в ячейку после неё, а не в код по ходу работы.

In [1]:
# Path — для удобной и переносимой работы с файловыми путями.
from pathlib import Path

# pandas — основной инструмент для чтения Excel и работы с таблицами.
import pandas as pd

In [2]:
# Имена файлов с исходными данными — если файлы переименуют, меняем только здесь.
CANDIDATE_FILE_NAME = "candidate_data.xlsx"
DICTIONARY_FILE_NAME = "data_dictionary.xlsx"

# Дата финального расчёта RecommendedOrder — задана условием задания.
CALCULATION_DATE = pd.Timestamp("2025-09-19")

# Целевой уровень сервиса для доработанной модели (эмпирический квантиль). Не задан в задании явно -
# 95% выбран как разумная отправная точка, чувствительность к этому значению проверяем отдельно.
SERVICE_LEVEL = 0.95

# Раздел 1. Baseline

## Шаг 1. Пути к данным

Определяем расположение исходных файлов (`candidate_data.xlsx`, `data_dictionary.xlsx`) относительно ноутбука и проверяем, что они на месте.

In [3]:
# Ноутбук лежит в notebooks/, поэтому поднимаемся на один уровень вверх, чтобы попасть в корень проекта.
project_dir = Path.cwd().parent

# Папка с исходными данными — на уровне корня проекта, не внутри notebooks/.
data_dir = project_dir / "data"

# Полные пути к обоим Excel-файлам, собранные из констант в ячейке параметров.
candidate_path = data_dir / CANDIDATE_FILE_NAME
dictionary_path = data_dir / DICTIONARY_FILE_NAME

# Проверяем, что оба файла реально существуют по этим путям, прежде чем пытаться их читать.
# Если assert сработает — сразу понятно, что не так, вместо непонятной ошибки на этапе чтения Excel.
assert candidate_path.exists(), f"Не найден файл: {candidate_path}"
assert dictionary_path.exists(), f"Не найден файл: {dictionary_path}"

print("candidate_data.xlsx найден:", candidate_path)
print("data_dictionary.xlsx найден:", dictionary_path)

candidate_data.xlsx найден: C:\Users\User\Desktop\clode folder\projects_code\тестовое от Генерации\reorder-model\data\candidate_data.xlsx
data_dictionary.xlsx найден: C:\Users\User\Desktop\clode folder\projects_code\тестовое от Генерации\reorder-model\data\data_dictionary.xlsx


## Шаг 2. Структура Excel-файлов

Смотрим список листов в обоих файлах, не загружая данные целиком.

In [4]:
# pd.ExcelFile открывает файл и читает его оглавление (список листов),
# но не загружает содержимое листов в память — это быстрее, чем сразу читать все данные.
candidate_excel = pd.ExcelFile(candidate_path)
dictionary_excel = pd.ExcelFile(dictionary_path)

print("Листы в candidate_data.xlsx:", candidate_excel.sheet_names)
print("Листы в data_dictionary.xlsx:", dictionary_excel.sheet_names)

Листы в candidate_data.xlsx: ['SKU', 'DailyHistory', 'OpenSupply', 'OrderCalendar']
Листы в data_dictionary.xlsx: ['Поля', 'Условия']


## Шаг 3. Словарь полей

Читаем лист "Поля" — описание всех колонок в данных, чтобы дальше корректно их интерпретировать.

In [5]:
# Читаем лист "Поля" целиком — он небольшой, это справочная таблица, а не данные для расчёта.
fields_df = pd.read_excel(dictionary_path, sheet_name="Поля")

# pd.set_option, чтобы длинные текстовые описания не обрезались "...".
pd.set_option("display.max_colwidth", None)

fields_df

,Лист,Поле,Единицы,Определение
0,SKU,SKU,Текст,Идентификатор позиции активного ассортимента. Один SKU — одна строка.
1,SKU,CurrentStock,Шт.,"Текущий остаток на дату расчёта 19.09.2025, до размещения заказа."
2,SKU,InTransit,Шт.,"Отгруженное, ещё не полученное количество. Сумма строк OpenSupply со Status = InTransit."
3,SKU,OpenPO,Шт.,"Размещённое, ещё не отгруженное количество. Сумма строк OpenSupply со Status = OpenPO."
4,SKU,LeadTimeDays,Календарные дни,Учебный срок нового заказа — 10 дней. Для уже размещённых поставок используйте ExpectedReceiptDate.
5,DailyHistory,Date,Дата,День наблюдения. Одна строка на SKU и календарный день.
6,DailyHistory,SKU,Текст,"Идентификатор, связанный с листом SKU."
7,DailyHistory,SalesQty,Шт.,Фактически выполненные продажи за указанный день. Возвраты не вычитаются.
8,DailyHistory,Остаток на конец дня,Шт.,Остаток на конец указанного дня. Пустая ячейка означает отсутствие наблюдения.
9,OpenSupply,PO_ID,Текст,Обезличенный идентификатор отдельной учебной поставки.


## Шаг 4. Условия задачи

Читаем лист "Условия" — явные допущения и параметры, заданные заказчиком (срок поставки, календарь и т.п.).

In [6]:
conditions_df = pd.read_excel(dictionary_path, sheet_name="Условия")
conditions_df

,Параметр,Значение
0,Дата расчёта,"19.09.2025, до размещения заказа"
1,История,19.06.2024–18.09.2025; 457 дней; 200 SKU
2,Гранулярность,SKU × календарный день
3,Конец дня,Столбец «Остаток на конец дня» относится к дню Date.
4,Единицы,Все количества — штуки; внутренние перемещения возможны поштучно.
5,Отсутствующие значения,Пустой остаток — наблюдения нет. Все SKU активного ассортимента остаются в итоговом расчёте.
6,История продаж,Фактически выполненные продажи. Исторического журнала неудовлетворённого спроса нет.
7,Учебные условия,"LeadTimeDays, календарь заказов и строки OpenSupply заданы для задания; это не реальные параметры компании."
8,Поставки,InTransit и OpenPO — разные состояния. Сводные количества SKU и строки OpenSupply описывают один набор поставок.
9,Исторические поставки,OpenSupply — снимок только на дату расчёта. Для backtesting требуется явно задать начальное состояние и правила моделирования.


## Шаг 5. Лист SKU

Загружаем таблицу активного ассортимента: текущие остатки, открытые поставки, срок поставки.

In [7]:
# Лист SKU — одна строка на позицию активного ассортимента (ожидаем 200 строк).
sku_df = pd.read_excel(candidate_path, sheet_name="SKU")

print("Форма таблицы (строки, колонки):", sku_df.shape)
print("Колонки:", list(sku_df.columns))
sku_df.head()

Форма таблицы (строки, колонки): (200, 5)
Колонки: ['SKU', 'CurrentStock', 'InTransit', 'OpenPO', 'LeadTimeDays']


,SKU,CurrentStock,InTransit,OpenPO,LeadTimeDays
0,TEST-0001,0,0,0,10
1,TEST-0002,3,0,1,10
2,TEST-0003,19,4,7,10
3,TEST-0004,0,0,0,10
4,TEST-0005,18,2,0,10


## Шаг 6. Лист DailyHistory

Загружаем дневную историю продаж и остатков — самая большая таблица (200 SKU × 457 дней).

In [8]:
history_df = pd.read_excel(candidate_path, sheet_name="DailyHistory")

print("Форма таблицы (строки, колонки):", history_df.shape)
print("Колонки:", list(history_df.columns))
print("Диапазон дат:", history_df["Date"].min(), "—", history_df["Date"].max())
history_df.head()

Форма таблицы (строки, колонки): (91400, 4)
Колонки: ['Date', 'SKU', 'SalesQty', 'Остаток на конец дня']
Диапазон дат: 2024-06-19 00:00:00 — 2025-09-18 00:00:00


,Date,SKU,SalesQty,Остаток на конец дня
0,2024-06-19,TEST-0001,0,2.0
1,2024-06-20,TEST-0001,0,2.0
2,2024-06-21,TEST-0001,0,2.0
3,2024-06-22,TEST-0001,0,2.0
4,2024-06-23,TEST-0001,0,2.0


## Шаг 7. Лист OpenSupply

Загружаем список уже размещённых поставок (InTransit/OpenPO) — снимок на дату расчёта.

In [9]:
open_supply_df = pd.read_excel(candidate_path, sheet_name="OpenSupply")

print("Форма таблицы (строки, колонки):", open_supply_df.shape)
print("Колонки:", list(open_supply_df.columns))
open_supply_df.head()

Форма таблицы (строки, колонки): (199, 6)
Колонки: ['PO_ID', 'SKU', 'Qty', 'Status', 'PlacedDate', 'ExpectedReceiptDate']


,PO_ID,SKU,Qty,Status,PlacedDate,ExpectedReceiptDate
0,PO-0002-1,TEST-0002,1,OpenPO,2025-09-15,2025-09-24
1,PO-0003-1,TEST-0003,4,InTransit,2025-09-14,2025-09-27
2,PO-0003-2,TEST-0003,7,OpenPO,2025-09-13,2025-10-04
3,PO-0005-1,TEST-0005,2,InTransit,2025-09-12,2025-10-03
4,PO-0006-1,TEST-0006,1,OpenPO,2025-09-11,2025-10-06


## Шаг 8. Лист OrderCalendar

Загружаем календарь размещения заказов — понадобится для расчёта protection period.

In [10]:
order_calendar_df = pd.read_excel(candidate_path, sheet_name="OrderCalendar")

print("Форма таблицы (строки, колонки):", order_calendar_df.shape)
print("Колонки:", list(order_calendar_df.columns))
order_calendar_df.head()

Форма таблицы (строки, колонки): (157, 3)
Колонки: ['OrderDate', 'NextOrderDate', 'OrderCycleDays']


,OrderDate,NextOrderDate,OrderCycleDays
0,2024-06-19,2024-06-21,2
1,2024-06-21,2024-06-26,5
2,2024-06-26,2024-06-28,2
3,2024-06-28,2024-07-03,5
4,2024-07-03,2024-07-05,2


## Шаг 9. Полнота календарной истории

Проверяем, что в `DailyHistory` ровно одна строка на каждую пару SKU-день, без пропущенных дней и без дублей.

In [11]:
# Считаем число строк, число уникальных пар SKU+Date и ожидаемое число (SKU x дни).
n_rows = len(history_df)
n_unique_pairs = history_df[["SKU", "Date"]].drop_duplicates().shape[0]
n_expected = sku_df["SKU"].nunique() * history_df["Date"].nunique()

print("Строк в DailyHistory:", n_rows)
print("Уникальных пар SKU+Date:", n_unique_pairs)
print("Ожидается (SKU x дни):", n_expected)
print("Дубликатов пар SKU+Date:", n_rows - n_unique_pairs)

Строк в DailyHistory: 91400
Уникальных пар SKU+Date: 91400
Ожидается (SKU x дни): 91400
Дубликатов пар SKU+Date: 0


## Шаг 10. Пропуски в данных

Считаем долю пропусков в `SalesQty` и в «Остаток на конец дня» (по словарю полей — пустой остаток значит «нет наблюдения», а не ноль).

In [12]:
missing_sales = history_df["SalesQty"].isna().sum()
missing_stock = history_df["Остаток на конец дня"].isna().sum()

print(f"Пропуски в SalesQty: {missing_sales} ({missing_sales / n_rows:.2%})")
print(f"Пропуски в 'Остаток на конец дня': {missing_stock} ({missing_stock / n_rows:.2%})")

Пропуски в SalesQty: 0 (0.00%)
Пропуски в 'Остаток на конец дня': 1489 (1.63%)


## Шаг 11. Активность продаж по SKU

Для каждого SKU считаем долю дней с продажами и суммарный спрос — чтобы понять, насколько распространён прерывистый спрос и есть ли SKU совсем без истории продаж.

In [13]:
# Группируем по SKU и считаем базовые показатели активности спроса.
sku_activity = history_df.groupby("SKU").agg(
    days_total=("SalesQty", "size"),
    days_with_sales=("SalesQty", lambda s: (s > 0).sum()),
    total_sales=("SalesQty", "sum"),
    mean_daily_sales=("SalesQty", "mean"),
).reset_index()

# Доля дней с продажами — ключевой индикатор прерывистости спроса.
sku_activity["share_days_with_sales"] = sku_activity["days_with_sales"] / sku_activity["days_total"]

print("SKU без единой продажи за всю историю:", (sku_activity["total_sales"] == 0).sum())
print("Медианная доля дней с продажами по SKU:", round(sku_activity["share_days_with_sales"].median(), 4))
print("SKU с долей дней с продажами < 10%:", (sku_activity["share_days_with_sales"] < 0.10).sum())

sku_activity.sort_values("total_sales").head(10)

SKU без единой продажи за всю историю: 18
Медианная доля дней с продажами по SKU: 0.0098
SKU с долей дней с продажами < 10%: 170


,SKU,days_total,days_with_sales,total_sales,mean_daily_sales,share_days_with_sales
7,TEST-0008,457,0,0,0.0,0.0
11,TEST-0012,457,0,0,0.0,0.0
29,TEST-0030,457,0,0,0.0,0.0
18,TEST-0019,457,0,0,0.0,0.0
49,TEST-0050,457,0,0,0.0,0.0
59,TEST-0060,457,0,0,0.0,0.0
43,TEST-0044,457,0,0,0.0,0.0
34,TEST-0035,457,0,0,0.0,0.0
60,TEST-0061,457,0,0,0.0,0.0
77,TEST-0078,457,0,0,0.0,0.0


## Шаг 12. Проверка на очевидные аномалии

Ищем отрицательные значения продаж/остатков и другие явно некорректные данные.

In [14]:
print("Отрицательные значения SalesQty:", (history_df["SalesQty"] < 0).sum())
print("Отрицательные значения остатка:", (history_df["Остаток на конец дня"] < 0).sum())
print("Максимальная продажа за день:", history_df["SalesQty"].max())
print("Максимальный остаток на конец дня:", history_df["Остаток на конец дня"].max())

Отрицательные значения SalesQty: 0
Отрицательные значения остатка: 0
Максимальная продажа за день: 150
Максимальный остаток на конец дня: 559.0


### Выводы и наблюдения

- Календарь наблюдений полный: 91 400 строк, дублей нет, пропущенных дней нет.
- `SalesQty` заполнен без единого пропуска — для наивной оценки спроса дополнительная очистка не требуется.
- В «Остаток на конец дня» — 1,63% пропусков (нет наблюдения, не ноль). Для baseline не критично (спрос оцениваем по `SalesQty`), но важно для бэктеста в разделе 2, где понадобится стартовое состояние запасов.
- Спрос по большинству SKU сильно прерывистый: медианная доля дней с продажами — **0,98%**, у 170 из 200 SKU (85%) доля дней с продажами меньше 10%. Наивное среднее по всей истории это переживёт, но будет давать грубую оценку — это ожидаемая точка для доработки в разделе 2, а не проблема, которую нужно решать прямо сейчас.
- **18 SKU (9%) не имеют вообще ни одной продажи за все 457 дней.** Для них наивная оценка спроса даст 0 — понадобится явный fallback и пометка в итоговой таблице (по условию задания такие позиции нельзя просто пропустить).
- Отрицательных значений и явных выбросов не найдено (максимум продаж за день — 150 шт., максимум остатка — 559 шт.) — данные в этой части чистые.

**Отдельный шаг «приведение данных в порядок» пропускаем** — для baseline он оказался бы пустым: `SalesQty` без пропусков/дублей/отрицательных значений, а два найденных момента (пропуски остатка, SKU без истории) — это не грязные данные, а два случая, которые нужно явно обработать логикой модели, а не почистить заранее.

## Шаг 13. Protection period — интервал до следующего заказа

Дата расчёта — это дата размещения заказа. Заказанное сегодня количество должно закрыть спрос не до следующей даты заказа, а до прихода *следующей за ней* поставки — поэтому берём интервал до следующего заказа из `OrderCalendar`, не хардкодим.

In [15]:
# Находим в календаре строку, где OrderDate — это наша дата расчёта.
calc_date_row = order_calendar_df[order_calendar_df["OrderDate"] == CALCULATION_DATE]

# Ожидаем ровно одну строку — на каждую разрешённую дату заказа календарь даёт один интервал до следующей.
assert len(calc_date_row) == 1, f"Ожидали одну строку календаря на {CALCULATION_DATE.date()}, нашли {len(calc_date_row)}"

# OrderCycleDays — сколько дней пройдёт до следующей возможности заказать (2 после среды, 5 после пятницы).
order_cycle_days = int(calc_date_row["OrderCycleDays"].iloc[0])
next_order_date = calc_date_row["NextOrderDate"].iloc[0]

print("Дата расчёта:", CALCULATION_DATE.date())
print("Следующая дата заказа:", next_order_date.date())
print("Интервал до следующего заказа (OrderCycleDays):", order_cycle_days, "дней")

Дата расчёта: 2025-09-19
Следующая дата заказа: 2025-09-24
Интервал до следующего заказа (OrderCycleDays): 5 дней


## Шаг 14. Protection period по каждому SKU

`LeadTimeDays` — колонка в `SKU`, а не глобальная константа, поэтому считаем protection period на уровне SKU (хотя по условию задания срок поставки для всех одинаковый, код не должен зависеть от этого совпадения).

In [16]:
# Начинаем сборку рабочей таблицы baseline_df с копии SKU, чтобы не менять исходный sku_df.
baseline_df = sku_df.copy()

# Protection period = срок поставки нового заказа (LeadTimeDays) + интервал до следующего заказа.
baseline_df["protection_period_days"] = baseline_df["LeadTimeDays"] + order_cycle_days

print("Уникальные значения LeadTimeDays:", sorted(baseline_df["LeadTimeDays"].unique()))
print("Уникальные значения protection_period_days:", sorted(baseline_df["protection_period_days"].unique()))

Уникальные значения LeadTimeDays: [np.int64(10)]
Уникальные значения protection_period_days: [np.int64(15)]


## Шаг 15. Наивная оценка спроса

Берём среднесуточный спрос по всей истории (уже посчитан в `sku_activity` на шаге 11 как `mean_daily_sales` — среднее по ВСЕМ дням, включая дни без продаж, что и есть корректная оценка интенсивности спроса). Умножаем на protection period.

In [17]:
# Подтягиваем среднесуточный спрос по SKU из таблицы активности (шаг 11).
baseline_df = baseline_df.merge(
    sku_activity[["SKU", "mean_daily_sales"]],
    on="SKU",
    how="left",
)

# Ожидаемый спрос за protection period — простое произведение, без поправок на дефицит.
baseline_df["demand_over_protection"] = baseline_df["mean_daily_sales"] * baseline_df["protection_period_days"]

print("SKU с нулевым ожидаемым спросом за protection period:", (baseline_df["demand_over_protection"] == 0).sum())
baseline_df[["SKU", "mean_daily_sales", "protection_period_days", "demand_over_protection"]].describe()

SKU с нулевым ожидаемым спросом за protection period: 18


,mean_daily_sales,protection_period_days,demand_over_protection
count,200.000000,200.0,200.000000
mean,0.374333,15.0,5.614989
std,1.131424,0.0,16.971359
min,0.000000,15.0,0.000000
25%,0.006565,15.0,0.098468
50%,0.031729,15.0,0.475930
75%,0.234683,15.0,3.520241
max,10.557987,15.0,158.369803


## Шаг 16. Order-up-to level (S) и Inventory Position — baseline

Baseline сознательно простой — без страхового запаса и без фильтрации поставок по горизонту прихода (это не забыто, а осознанно отложено: baseline должен быть "наивной" точкой сравнения для более аккуратной модели в разделе 2, а не черновиком финальной модели).

- `S = demand_over_protection` (без safety stock).
- `InventoryPosition = CurrentStock + InTransit + OpenPO` (без фильтра по дате прихода).

In [18]:
# Order-up-to level для baseline — без страхового запаса, чистое ожидание спроса.
baseline_df["S_baseline"] = baseline_df["demand_over_protection"]

# Позиция запаса — текущий остаток плюс всё, что уже в пути или заказано.
baseline_df["inventory_position"] = (
    baseline_df["CurrentStock"] + baseline_df["InTransit"] + baseline_df["OpenPO"]
)

baseline_df[["SKU", "S_baseline", "inventory_position"]].describe()

,S_baseline,inventory_position
count,200.000000,200.000000
mean,5.614989,15.515000
std,16.971359,33.771621
min,0.000000,0.000000
25%,0.098468,2.000000
50%,0.475930,4.000000
75%,3.520241,13.000000
max,158.369803,236.000000


## Шаг 17. RecommendedOrder — baseline

`RecommendedOrder = max(0, round(S - InventoryPosition))` — целое, неотрицательное, как требует задание.

In [19]:
# round() до целого, затем max(0, ...) — неотрицательное количество.
gap = baseline_df["S_baseline"] - baseline_df["inventory_position"]
baseline_df["RecommendedOrder_baseline"] = gap.round().clip(lower=0).astype(int)

print("Строк в baseline_df:", len(baseline_df))
print("Все 200 SKU на месте:", set(baseline_df["SKU"]) == set(sku_df["SKU"]))
print("Суммарный объём заказа по baseline:", baseline_df["RecommendedOrder_baseline"].sum(), "шт.")
print("SKU с RecommendedOrder > 0:", (baseline_df["RecommendedOrder_baseline"] > 0).sum())

baseline_df[["SKU", "S_baseline", "inventory_position", "RecommendedOrder_baseline"]].sort_values(
    "RecommendedOrder_baseline", ascending=False
).head(10)

Строк в baseline_df: 200
Все 200 SKU на месте: True
Суммарный объём заказа по baseline: 97 шт.
SKU с RecommendedOrder > 0: 18


,SKU,S_baseline,inventory_position,RecommendedOrder_baseline
135,TEST-0136,22.910284,0,23
24,TEST-0025,96.695842,85,12
48,TEST-0049,9.518600,0,10
33,TEST-0034,7.483589,0,7
81,TEST-0082,15.689278,9,7
108,TEST-0109,6.630197,0,7
51,TEST-0052,7.385120,0,7
129,TEST-0130,7.582057,3,5
162,TEST-0163,4.135667,0,4
90,TEST-0091,3.971554,0,4


### Выводы и наблюдения

- Protection period на 19.09.2025 = 10 (лид-тайм) + 5 (интервал до следующего заказа, из календаря) = **15 дней**, одинаков для всех SKU (`LeadTimeDays` в данных везде равен 10 — но код это не предполагает заранее, а вычисляет).
- Baseline-формула прогнана на всех 200 SKU без ошибок, `RecommendedOrder` — целое и неотрицательное у всех строк.
- Суммарный объём заказа по baseline — **97 шт.**, но только у **18 из 200 SKU** (9%) получилось `RecommendedOrder > 0`. Это ожидаемо, а не баг: без страхового запаса заказ появляется только там, где текущей позиции запаса не хватает даже на голый средний спрос за 15 дней — для позиций с прерывистым спросом это редкий случай.
- 18 SKU без истории продаж (найдены в EDA) получили `RecommendedOrder_baseline = 0` — это корректное поведение формулы (нулевой спрос → нулевая рекомендация), но по условию задания для таких позиций нужна явная пометка в итоговой таблице. Формулу это не меняет, оставляем на раздел 3 (там же, где собирается финальная таблица).
- Baseline осознанно **не включает**: страховой запас, поправку на дефицит (censored demand), фильтрацию поставок по горизонту прихода. Это не недоработка, а точка сравнения — та же формула (`S_baseline`, `inventory_position`) переиспользуется без изменений как простой baseline при сравнении с доработанной моделью в разделе 2.

# Раздел 2. Бэктесты и отладка

## Шаг 18. Выбор контрольных дат

Берём даты заказа из `OrderCalendar` (кроме самой даты расчёта 19.09.2025 — она зарезервирована под финальный расчёт в разделе 3). Валидна дата, если после неё в истории есть данные минимум на весь её protection period вперёд — иначе нам не с чем будет сравнить рекомендацию (не увидим реальный спрос за этот период). Из валидных берём 8 последних — не нужно перебирать все 156, для проверки модели достаточно.

In [20]:
# На шаге 14 подтвердили, что LeadTimeDays одинаков у всех SKU — фиксируем это явной проверкой
# (а не молчаливым предположением) и используем как скаляр для расчётов на уровне календаря.
assert sku_df["LeadTimeDays"].nunique() == 1, "LeadTimeDays неоднороден - protection period нельзя считать на уровне календаря одним числом"
LEAD_TIME_DAYS = int(sku_df["LeadTimeDays"].iloc[0])

In [21]:
# Последний доступный день фактической истории — дальше него сравнивать не с чем.
last_history_date = history_df["Date"].max()

# protection period одинаков для всех SKU (LeadTimeDays константа), поэтому считаем его на уровне календаря,
# а не на уровне SKU, как в baseline — тут это одно число на дату, а не на строку.
calendar_check = order_calendar_df.copy()
calendar_check["protection_period_days"] = calendar_check["OrderCycleDays"] + LEAD_TIME_DAYS
calendar_check["last_covered_date"] = calendar_check["OrderDate"] + pd.to_timedelta(
    calendar_check["protection_period_days"], unit="D"
)

# Валидные даты: не сама дата расчёта, и после них в истории есть данные на весь protection period.
is_not_calc_date = calendar_check["OrderDate"] != CALCULATION_DATE
has_enough_future_history = calendar_check["last_covered_date"] <= last_history_date
valid_control_dates = calendar_check[is_not_calc_date & has_enough_future_history]

# Берём 8 последних валидных дат — они ближе всего по времени к дате расчёта.
N_CONTROL_DATES = 8
control_dates_df = valid_control_dates.sort_values("OrderDate").tail(N_CONTROL_DATES).reset_index(drop=True)

print("Всего дат в календаре:", len(order_calendar_df))
print("Валидных дат (хватает будущей истории):", len(valid_control_dates))
print("Выбрано контрольных дат:", len(control_dates_df))
control_dates_df[["OrderDate", "OrderCycleDays", "protection_period_days", "last_covered_date"]]

Всего дат в календаре: 157
Валидных дат (хватает будущей истории): 127
Выбрано контрольных дат: 8


,OrderDate,OrderCycleDays,protection_period_days,last_covered_date
0,2025-08-08,5,15,2025-08-23
1,2025-08-13,2,12,2025-08-25
2,2025-08-15,5,15,2025-08-30
3,2025-08-20,2,12,2025-09-01
4,2025-08-22,5,15,2025-09-06
5,2025-08-27,2,12,2025-09-08
6,2025-08-29,5,15,2025-09-13
7,2025-09-03,2,12,2025-09-15


## Шаг 19. Допущения бэктеста

Исторического журнала заказов и открытых поставок нет — снимок `OpenSupply` есть только на 19.09.2025. Поэтому для каждой контрольной даты явно задаём:

1. **Стартовая позиция запаса** = фактический остаток на конец дня, предшествующего контрольной дате (`Остаток на конец дня` за `T − 1`), из реальной истории. `InTransit`/`OpenPO` на исторические даты принимаем за 0 — у нас нет данных, что реально было в пути в прошлом, а не потому что их не было.
2. **Оценка спроса** считается только по истории **строго до** контрольной даты — без утечки будущего в параметры модели.
3. **Проверка решения — по сумме за весь protection period, без разбиения на "до/после прихода заказа".** `shortfall = max(0, факт_спрос_за_период − (inventory_position + RecommendedOrder))`. Осознанно не делим период на фазы: решение о размере заказа принимается один раз, до начала периода, и никак не может повлиять на то, случится ли дефицит именно в первые `LeadTimeDays` дней, пока заказ ещё в пути — сколько бы мы ни заказали, это не изменится. Разбиение на фазы не изменило бы наше решение сегодня, только усложнило бы проверку. **Ограничение, которое это даёт:** метрика может переоценивать реальный сервис у SKU, чей спрос сильно смещён к началу периода (до прихода заказа) — дефицит там реален, но по сумме за весь период может "спрятаться" за профицитом во второй половине. Это отмечено как известное ограничение, а не скрытая ошибка.

## Шаг 20. Остаток с заполнением пропусков (для подстановки на любую дату)

На EDA нашли 1,63% пропусков в «Остаток на конец дня» — для итогового расчёта (раздел 3) это было не нужно, но для бэктеста нужно уметь взять остаток на *любую* историческую дату, в т.ч. там, где наблюдения нет. Заполняем пропуски последним известным значением по каждому SKU (вперёд по времени) — самое простое и защитимое допущение: "остаток не наблюдали, но и явных признаков, что он изменился, тоже нет".

In [22]:
# Сортируем по SKU и дате — обязательное условие для корректного forward-fill внутри каждого SKU.
history_sorted = history_df.sort_values(["SKU", "Date"]).copy()

# ffill внутри каждой группы SKU: пропуск заполняется последним известным остатком этого же SKU.
history_sorted["stock_filled"] = history_sorted.groupby("SKU")["Остаток на конец дня"].ffill()

# Сколько осталось пропусков после ffill — должны остаться только самые первые дни истории SKU,
# если у него вообще нет ни одного наблюдения остатка до какой-то точки.
remaining_missing = history_sorted["stock_filled"].isna().sum()
print("Пропусков после ffill:", remaining_missing, f"({remaining_missing / len(history_sorted):.3%})")

Пропусков после ffill: 1093 (1.196%)


## Шаг 21. Функция: protection period на произвольную дату

Обобщаем логику шага 13 (там она считалась разово под `CALCULATION_DATE`) в функцию — она понадобится для каждой из 8 контрольных дат.

In [23]:
def get_protection_period_days(order_date):
    """Protection period = LeadTimeDays + интервал до следующего заказа из OrderCalendar."""
    # Находим строку календаря для этой даты заказа — та же логика, что на шаге 13.
    row = order_calendar_df[order_calendar_df["OrderDate"] == order_date]
    assert len(row) == 1, f"Ожидали одну строку календаря на {order_date}, нашли {len(row)}"
    return int(row["OrderCycleDays"].iloc[0]) + LEAD_TIME_DAYS


# Проверяем функцию на уже известном результате шага 13 (19.09.2025 -> 15 дней).
assert get_protection_period_days(CALCULATION_DATE) == 15
print("Функция проверена на дате расчёта: 15 дней — совпадает с шагом 13.")

Функция проверена на дате расчёта: 15 дней — совпадает с шагом 13.


## Шаг 22. Функция: наивная оценка спроса по данным до даты

Ключевое отличие от baseline: там усредняли по всей истории, здесь — только по истории, доступной **до** конкретной даты решения (иначе в бэктест утечёт будущее).

In [24]:
def get_naive_demand_estimate(as_of_date):
    """Средний дневной спрос по каждому SKU, посчитанный только по дням строго до as_of_date."""
    # Берём только "прошлое" относительно даты решения — без подглядывания в будущее.
    past_history = history_df[history_df["Date"] < as_of_date]
    return past_history.groupby("SKU")["SalesQty"].mean().rename("mean_daily_sales")


# Проверяем на дате расчёта: там мы использовали среднее по ВСЕЙ истории (в ней и так нет дней после 18.09.2025).
check = get_naive_demand_estimate(CALCULATION_DATE)
comparison = sku_activity.set_index("SKU")["mean_daily_sales"].compare(check.reindex(sku_activity["SKU"]))
print("Расхождений с шагом 15 (полная история):", len(comparison))

Расхождений с шагом 15 (полная история): 0


## Шаг 23. Функция: позиция запаса на историческую дату

По допущению шага 19 — фактический остаток на конец предыдущего дня, `InTransit`/`OpenPO` = 0 (для исторических дат этих данных нет).

In [25]:
def get_inventory_position_asof(control_date):
    """Стартовая позиция запаса = остаток (с ffill) на конец дня перед control_date. InTransit/OpenPO = 0."""
    prev_day = control_date - pd.Timedelta(days=1)
    day_slice = history_sorted[history_sorted["Date"] == prev_day]
    position = day_slice.set_index("SKU")["stock_filled"].rename("inventory_position")

    # У части SKU нет вообще ни одного наблюдения остатка до этой даты (ffill заполнить нечем) —
    # для них считаем позицию запаса неизвестной, поэтому явно нулевой, а не выдумываем число.
    n_missing = position.isna().sum()
    if n_missing:
        position = position.fillna(0.0)
    return position, n_missing


# Проверяем на первой контрольной дате — сколько SKU потребовали fallback на 0.
test_position, test_missing = get_inventory_position_asof(control_dates_df["OrderDate"].iloc[0])
print("SKU с известной позицией запаса:", (test_position.notna()).sum() - test_missing, "из 200")
print("SKU, для которых позицию запаса пришлось принять за 0 (нет ни одного наблюдения):", test_missing)

SKU с известной позицией запаса: 198 из 200
SKU, для которых позицию запаса пришлось принять за 0 (нет ни одного наблюдения): 2


## Шаг 24. Решения baseline-модели по всем 8 контрольным датам

Собираем те же функции в одну таблицу решений: 200 SKU × 8 дат = 1600 строк. Формула та же, что в разделе 1 (шаги 15-17), просто применённая многократно, на разные даты, вместо одного разового расчёта.

In [26]:
decision_rows = []

# Проходим по каждой из 8 контрольных дат и считаем решение для всех 200 SKU разом.
for _, cal_row in control_dates_df.iterrows():
    control_date = cal_row["OrderDate"]
    protection_period_days = int(cal_row["protection_period_days"])

    demand_estimate = get_naive_demand_estimate(control_date)
    inventory_position, _ = get_inventory_position_asof(control_date)

    decision = pd.DataFrame({"SKU": sku_df["SKU"]})
    decision["control_date"] = control_date
    decision["protection_period_days"] = protection_period_days
    decision = decision.merge(demand_estimate, on="SKU", how="left")
    decision = decision.merge(inventory_position, on="SKU", how="left")

    decision["S"] = decision["mean_daily_sales"] * decision["protection_period_days"]
    gap = decision["S"] - decision["inventory_position"]
    decision["RecommendedOrder"] = gap.round().clip(lower=0).astype(int)

    decision_rows.append(decision)

# Склеиваем решения по всем датам в одну длинную таблицу.
baseline_decisions_df = pd.concat(decision_rows, ignore_index=True)

print("Строк (SKU x даты):", len(baseline_decisions_df))
print("Пропуски в ключевых колонках:", baseline_decisions_df[["mean_daily_sales", "inventory_position", "RecommendedOrder"]].isna().sum().to_dict())
baseline_decisions_df.head()

Строк (SKU x даты): 1600


Пропуски в ключевых колонках:

 {'mean_daily_sales': 0, 'inventory_position': 0, 'RecommendedOrder': 0}


,SKU,control_date,protection_period_days,mean_daily_sales,inventory_position,S,RecommendedOrder
0,TEST-0001,2025-08-08,15,0.004819,0.0,0.072289,0
1,TEST-0002,2025-08-08,15,0.000000,5.0,0.000000,0
2,TEST-0003,2025-08-08,15,0.371084,19.0,5.566265,0
3,TEST-0004,2025-08-08,15,0.028916,0.0,0.433735,0
4,TEST-0005,2025-08-08,15,0.221687,14.0,3.325301,0


## Шаг 25. Фактический спрос за protection period

Для каждой (SKU, дата) считаем реальные продажи с `control_date` по `control_date + protection_period_days − 1` включительно — это то, с чем будем сравнивать решение модели.

In [27]:
def get_actual_demand_over_window(control_date, protection_period_days):
    """Сумма фактических продаж по каждому SKU за [control_date, control_date + protection_period_days - 1]."""
    window_end = control_date + pd.Timedelta(days=protection_period_days - 1)
    window = history_df[(history_df["Date"] >= control_date) & (history_df["Date"] <= window_end)]
    return window.groupby("SKU")["SalesQty"].sum().rename("actual_demand")


# Считаем фактический спрос для каждой контрольной даты и сразу подклеиваем к таблице решений.
actual_demand_parts = []
for _, cal_row in control_dates_df.iterrows():
    control_date = cal_row["OrderDate"]
    protection_period_days = int(cal_row["protection_period_days"])
    part = get_actual_demand_over_window(control_date, protection_period_days).reset_index()
    part["control_date"] = control_date
    actual_demand_parts.append(part)

actual_demand_df = pd.concat(actual_demand_parts, ignore_index=True)

baseline_decisions_df = baseline_decisions_df.merge(actual_demand_df, on=["SKU", "control_date"], how="left")

print("Пропуски в actual_demand:", baseline_decisions_df["actual_demand"].isna().sum())
baseline_decisions_df[["SKU", "control_date", "protection_period_days", "RecommendedOrder", "actual_demand"]].head()

Пропуски в actual_demand: 0


,SKU,control_date,protection_period_days,RecommendedOrder,actual_demand
0,TEST-0001,2025-08-08,15,0,0
1,TEST-0002,2025-08-08,15,0,2
2,TEST-0003,2025-08-08,15,0,5
3,TEST-0004,2025-08-08,15,0,0
4,TEST-0005,2025-08-08,15,0,5


## Шаг 26. Дефицит, излишек, успешность цикла

Как договорились: без разбиения на фазы, по сумме за весь period.

In [28]:
# Сколько всего будет доступно за period: то, что уже есть, плюс то, что закажем.
baseline_decisions_df["available_to_cover"] = (
    baseline_decisions_df["inventory_position"] + baseline_decisions_df["RecommendedOrder"]
)

# Дефицит — если факт спроса превысил доступное; излишек — если наоборот.
baseline_decisions_df["shortfall"] = (
    baseline_decisions_df["actual_demand"] - baseline_decisions_df["available_to_cover"]
).clip(lower=0)
baseline_decisions_df["excess"] = (
    baseline_decisions_df["available_to_cover"] - baseline_decisions_df["actual_demand"]
).clip(lower=0)

# Цикл успешен, если дефицита не было вообще.
baseline_decisions_df["cycle_success"] = baseline_decisions_df["shortfall"] == 0

baseline_decisions_df[["SKU", "control_date", "available_to_cover", "actual_demand", "shortfall", "excess", "cycle_success"]].head()

,SKU,control_date,available_to_cover,actual_demand,shortfall,excess,cycle_success
0,TEST-0001,2025-08-08,0.0,0,0.0,0.0,True
1,TEST-0002,2025-08-08,5.0,2,0.0,3.0,True
2,TEST-0003,2025-08-08,19.0,5,0.0,14.0,True
3,TEST-0004,2025-08-08,0.0,0,0.0,0.0,True
4,TEST-0005,2025-08-08,14.0,5,0.0,9.0,True


## Шаг 27. Метрики baseline по 8 контрольным датам

Cycle Service Level, Fill Rate, средний/максимальный запас, суммарный дефицит/излишек — определения из обсуждения выше.

In [29]:
def compute_metrics(decisions_df, model_name):
    """Сводная строка метрик по таблице решений — переиспользуем для baseline и для доработанной модели."""
    return pd.DataFrame([{
        "model": model_name,
        "cycle_service_level": decisions_df["cycle_success"].mean(),
        "fill_rate": 1 - decisions_df["shortfall"].sum() / decisions_df["actual_demand"].sum(),
        "avg_inventory": decisions_df["available_to_cover"].mean(),
        "max_inventory": decisions_df["available_to_cover"].max(),
        "total_shortfall": decisions_df["shortfall"].sum(),
        "total_excess": decisions_df["excess"].sum(),
    }])


baseline_metrics_df = compute_metrics(baseline_decisions_df, "Baseline")
baseline_metrics_df

,model,cycle_service_level,fill_rate,avg_inventory,max_inventory,total_shortfall,total_excess
0,Baseline,0.950625,0.851643,14.285,326.0,1291.0,15445.0


### Выводы и наблюдения

- Baseline по 8 контрольным датам (1600 решений SKU×дата): **Cycle Service Level 95,1%**, но **Fill Rate только 85,2%**.
- Разрыв между метриками — не ошибка, а ожидаемое следствие сильно прерывистого спроса: у большинства (SKU, дата) фактический спрос почти нулевой, поэтому "не уйти в дефицит" — тривиально легко, отсюда высокий Cycle Service Level. А редкие решения с реально заметным спросом промахиваются на существенный объём (нет страхового запаса) — и именно они утягивают вниз Fill Rate, который взвешен по штукам, а не по количеству решений.
- Средний запас — всего 14,3 шт., но максимум доходит до 326 — сильная неоднородность между "спокойными" и "объёмными" SKU.
- Суммарный излишек (15 445 шт.) почти в 12 раз больше суммарного дефицита (1 291 шт.) — типичная картина для baseline без страхового запаса на лампи-спросе: у "тихих" SKU почти любой остаток избыточен относительно их мизерного среднего спроса, а у "объёмных" SKU средний спрос как оценка систематически недооценивает реальные всплески.

Это прямая мотивация для доработки: следующий шаг — заменить чистое среднее на оценку с учётом изменчивости спроса (страховой запас), чтобы целенаправленно закрыть именно те 14,8% дефицита в штуках, не раздувая и без того избыточный запас у стабильных SKU.

## Шаг 28. Доработанная модель: страховой запас через эмпирический квантиль

Baseline использовал только среднее — поэтому не защищён от всплесков спроса. Классическая формула страхового запаса (`z × sigma × sqrt(period)`) предполагает нормальное распределение спроса — а в EDA мы выяснили, что спрос у большинства SKU сильно прерывистый (лампи), это предположение для него не подходит и легко даёт странные/отрицательные оценки при низком среднем и высокой дисперсии.

Вместо этого берём **эмпирический квантиль** — без предположений о форме распределения:

1. По каждому SKU берём историю **до** даты решения (без утечки будущего, как и раньше).
2. Считаем скользящую сумму продаж в окнах длиной `protection_period_days` (сколько реально продавалось за такие периоды раньше).
3. `S = квантиль этого распределения` на выбранном уровне сервиса (например, 95-й процентиль) — то есть "такого спроса за period, как правило, не превышали в 95% случаев из прошлого".

Это одновременно и оценка среднего, и запас на изменчивость — без разделения на два отдельных слагаемых, и без привязки к нормальному распределению.

In [30]:
def get_empirical_S(as_of_date, protection_period_days, service_level):
    """S = квантиль скользящих сумм спроса за окна длиной protection_period_days, по истории до as_of_date."""
    past_history = history_df[history_df["Date"] < as_of_date].sort_values(["SKU", "Date"])

    # Скользящая сумма продаж в окне protection_period_days, отдельно для каждого SKU.
    rolling_sum = (
        past_history.groupby("SKU")["SalesQty"]
        .rolling(protection_period_days)
        .sum()
        .reset_index(level=0)  # возвращаем SKU из индекса в колонку
    )

    # Квантиль по каждому SKU от полученного распределения скользящих сумм.
    S = rolling_sum.groupby("SKU")["SalesQty"].quantile(service_level).rename("S")
    S = S.reindex(sku_df["SKU"])

    # NaN здесь означает "нет ни одного полного окна" (реально мало истории) - это другой случай,
    # чем "окно есть, но квантиль честно получился нулевым" (очень редкие продажи). Разделяем их явно.
    n_no_window = S.isna().sum()
    return S.fillna(0.0), n_no_window


# Проверяем на первой контрольной дате.
test_S, test_no_window = get_empirical_S(control_dates_df["OrderDate"].iloc[0], 15, SERVICE_LEVEL)
print("SKU без полного окна истории (данных объективно мало):", test_no_window, "из 200")
print("SKU с честно нулевым квантилем (окно есть, но продаж почти не было):", (test_S == 0).sum() - test_no_window)
test_S.describe()

SKU без полного окна истории (данных объективно мало): 0 из 200
SKU с честно нулевым квантилем (окно есть, но продаж почти не было): 61


count    200.000000
mean      15.730000
std       40.409177
min        0.000000
25%        0.000000
50%        3.000000
75%       11.250000
max      362.000000
Name: S, dtype: float64

## Шаг 29. Решения доработанной модели по тем же 8 датам

Та же механика, что и для baseline (шаг 24) — только `S` теперь из `get_empirical_S`, а не из простого среднего. `inventory_position` и сравнение с фактом переиспользуем без изменений.

In [31]:
def build_decisions(control_dates_df, S_func, **S_kwargs):
    """Общая сборка таблицы решений SKU x даты - принимает функцию расчёта S, чтобы не дублировать код
    между baseline и доработанной моделью (отличаются только тем, как считается S)."""
    rows = []
    for _, cal_row in control_dates_df.iterrows():
        control_date = cal_row["OrderDate"]
        protection_period_days = int(cal_row["protection_period_days"])

        S = S_func(control_date, protection_period_days, **S_kwargs)
        if isinstance(S, tuple):  # get_empirical_S возвращает (S, n_no_window)
            S = S[0]

        inventory_position, _ = get_inventory_position_asof(control_date)

        decision = pd.DataFrame({"SKU": sku_df["SKU"]})
        decision["control_date"] = control_date
        decision["protection_period_days"] = protection_period_days
        decision = decision.merge(S.rename("S"), on="SKU", how="left")
        decision = decision.merge(inventory_position, on="SKU", how="left")

        gap = decision["S"] - decision["inventory_position"]
        decision["RecommendedOrder"] = gap.round().clip(lower=0).astype(int)
        rows.append(decision)

    decisions_df = pd.concat(rows, ignore_index=True)
    decisions_df = decisions_df.merge(actual_demand_df, on=["SKU", "control_date"], how="left")

    decisions_df["available_to_cover"] = decisions_df["inventory_position"] + decisions_df["RecommendedOrder"]
    decisions_df["shortfall"] = (decisions_df["actual_demand"] - decisions_df["available_to_cover"]).clip(lower=0)
    decisions_df["excess"] = (decisions_df["available_to_cover"] - decisions_df["actual_demand"]).clip(lower=0)
    decisions_df["cycle_success"] = decisions_df["shortfall"] == 0
    return decisions_df


improved_decisions_df = build_decisions(control_dates_df, get_empirical_S, service_level=SERVICE_LEVEL)

print("Строк:", len(improved_decisions_df))
print("Пропуски:", improved_decisions_df[["S", "inventory_position", "actual_demand"]].isna().sum().to_dict())
improved_decisions_df.head()

Строк: 1600
Пропуски: {'S': 0, 'inventory_position': 0, 'actual_demand': 0}


,SKU,control_date,protection_period_days,S,inventory_position,RecommendedOrder,actual_demand,available_to_cover,shortfall,excess,cycle_success
0,TEST-0001,2025-08-08,15,0.0,0.0,0,0,0.0,0.0,0.0,True
1,TEST-0002,2025-08-08,15,0.0,5.0,0,2,5.0,0.0,3.0,True
2,TEST-0003,2025-08-08,15,14.0,19.0,0,5,19.0,0.0,14.0,True
3,TEST-0004,2025-08-08,15,4.0,0.0,4,0,4.0,0.0,4.0,True
4,TEST-0005,2025-08-08,15,7.0,14.0,0,5,14.0,0.0,9.0,True


## Шаг 30. Сравнение с baseline

Считаем метрики доработанной модели той же функцией `compute_metrics` (шаг 27) и кладём рядом с baseline в одну таблицу.

In [32]:
improved_metrics_df = compute_metrics(improved_decisions_df, f"Доработанная модель (SL={SERVICE_LEVEL:.0%})")

comparison_df = pd.concat([baseline_metrics_df, improved_metrics_df], ignore_index=True)
comparison_df

,model,cycle_service_level,fill_rate,avg_inventory,max_inventory,total_shortfall,total_excess
0,Baseline,0.950625,0.851643,14.285000,326.0,1291.0,15445.0
1,Доработанная модель (SL=95%),0.980000,0.934728,19.295625,362.0,568.0,22739.0


### Выводы и наблюдения

- Доработанная модель (эмпирический квантиль, SL=95%) против baseline: **Cycle Service Level 95,1% → 98,0%**, **Fill Rate 85,2% → 93,5%** — заметное улучшение по обеим метрикам, особенно по Fill Rate (именно там, где у baseline была самая слабая точка).
- Цена улучшения — средний запас вырос с 14,3 до 19,3 шт. (+35%), суммарный излишек — с 15 445 до 22 739 шт. Это ожидаемый и честный компромисс "сервис против запаса", а не бесплатное улучшение.
- Суммарный дефицит упал более чем вдвое (1291 → 568 шт.) — модель действительно закрывает именно ту проблему, которую нашли в baseline (недооценка всплесков спроса), а не улучшает метрики за счёт общего раздувания заказа по всем SKU подряд.

## Шаг 31. Чувствительность к уровню сервиса

Пересчитываем доработанную модель при Service Level 90/95/98% — смотрим, как меняются метрики. Переиспользуем `build_decisions` и `compute_metrics`, ничего нового писать не нужно.

In [33]:
sensitivity_rows = [baseline_metrics_df]

# Прогоняем доработанную модель при трёх уровнях сервиса - переиспользуем те же функции.
for sl in [0.90, 0.95, 0.98]:
    decisions = build_decisions(control_dates_df, get_empirical_S, service_level=sl)
    metrics = compute_metrics(decisions, f"SL={sl:.0%}")
    sensitivity_rows.append(metrics)

sensitivity_df = pd.concat(sensitivity_rows, ignore_index=True)
sensitivity_df

,model,cycle_service_level,fill_rate,avg_inventory,max_inventory,total_shortfall,total_excess
0,Baseline,0.950625,0.851643,14.285000,326.0,1291.0,15445.0
1,SL=90%,0.971250,0.923811,17.229375,326.0,663.0,19528.0
2,SL=95%,0.980000,0.934728,19.295625,362.0,568.0,22739.0
3,SL=98%,0.986875,0.956217,21.960000,414.0,381.0,26815.0


### Выводы по разделу 2

**Чувствительность:** рост Service Level с 90% до 98% даёт монотонный (без скачков и аномалий) компромисс — Fill Rate растёт с 92,4% до 95,6%, средний запас растёт с 17,2 до 22,0 шт. Модель ведёт себя предсказуемо во всём диапазоне.

**Выбор итогового уровня сервиса — 95%.** Задание не фиксирует целевой Service Level числом явно, поэтому выбор обоснован из самой чувствительности: переход с 90% на 95% даёт заметный прирост Fill Rate (+1,1 п.п.) за умеренную цену (+2,1 шт. среднего запаса), а переход с 95% на 98% даёт куда меньший прирост (+2,1 п.п. Fill Rate) за похожую по масштабу цену (+2,7 шт.) — то есть после 95% начинается заметное снижение отдачи от дальнейшего роста запаса. 95% — точка, где выигрыш ещё ощутимо опережает издержки.

**Ограничения методологии бэктеста (честно, не скрываем):**
- Проверка решения — по сумме за весь период, без учёта, что заказ физически появляется только после lead time (шаг 19) — может переоценивать сервис у SKU со спросом, смещённым к началу периода. Осознанный выбор: размер сегодняшнего заказа всё равно не может повлиять на дефицит в первые `LeadTimeDays` дней (заказ ещё физически не может прийти раньше), поэтому разбиение на фазы не изменило бы само решение, а только усложнило бы проверку.
- `InTransit`/`OpenPO` = 0 для всех исторических контрольных дат — потому что исторического журнала поставок нет физически, а не потому что их не было на самом деле. Значит, метрики бэктеста, вероятно, консервативны (может недооценивать реальный исторический сервис, если поставки в пути реально были).
- Всего 8 контрольных дат (08.08–03.09.2025) — осознанно небольшая выборка, покрывающая примерно последний месяц перед датой расчёта, а не всю историю: расширение выборки не меняет структуру вывода, но пропорционально увеличивает время расчёта.

Аномалии и ограничения самих данных (пропуски, SKU без истории и т.п.) зафиксированы в разделе 1 (EDA) — здесь не дублируем.

# Раздел 3. Заказ